# Inference on M81 Satellites (SQL → MLP)

This notebook loads the trained MLP model from `02_mlp_training_on_tng_features.ipynb` and applies it to the M81 satellite system.

Pipeline:
- Query satellite features from a local SQLite database (`satellites.db`, table: `m81_satellites`) using SQL
- Preprocess features consistently with the training setup
- Load the saved PyTorch checkpoint and run inference
- Save predictions and generate summary plots/tables

Data source:
- SQLite database: `satellites.db`
- Table: `m81_satellites`

Note on satellite count (N):
- The uploaded version assumes **N = 31** satellites (M81 sample used in this repo).
- To run the pipeline with a different N, update the corresponding parameter in **both** Notebook 02 (training) and Notebook 03 (inference) so that the model input dimension matches.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import sqlite3
import pandas as pd

In [2]:
conn = sqlite3.connect("satellites.db")

# SQL
read =  """
    SELECT name, log10_LK, R_proj_kpc, delta_v_kms
    FROM m81_satellites
    WHERE log10_LK > 0
    ORDER BY R_proj_kpc;
    """
df = pd.read_sql(read, conn)

# to np.array
names  = df["name"].to_numpy()
logL_K = df["log10_LK"].to_numpy(dtype=float)
Rp_kpc = df["R_proj_kpc"].to_numpy(dtype=float)
deltaV = df["delta_v_kms"].to_numpy(dtype=float)

n_subs = len(deltaV)
print(f"\nRemaining N_sub = {n_subs}")
pd.read_sql("""SELECT * FROM m""", conn)


Remaining N_sub = 31


,name,log10_LK,R_proj_kpc,delta_v_kms
0,Holm IX,7.75,12,88
1,BK3N,6.12,12,-3
2,JKB3,5.63,17,94
3,A0952+69,6.87,18,138
4,Clump I,5.57,25,-129
5,KDG 61,8.11,32,256
6,KDG 61em,5.98,32,151
7,D0959+68,6.44,35,-150
8,Clump III,5.57,40,-85
9,M82,10.59,40,224


In [3]:
G_dyn = 4.302e-9
C_rad = 16.0 / np.pi
t_U = 13.8                        # Gyr, age of the Universe
G_cosmo = 4.498638185699749e-15   # Mpc^3 / (Msun Gyr^2)


# -------------------------------
# Convert projected radius to Mpc and prepare LOS velocity
# -------------------------------
R_proj_Mpc = Rp_kpc / 1000.0
v_los = deltaV


# -------------------------------
# Projected Mass Estimator (PME/TME)
# M = (16 / (π G)) (1/N) Σ(v_los^2 R_proj)
# -------------------------------
M_TME = (C_rad / (G_dyn * n_subs)) * np.sum((v_los**2) * R_proj_Mpc)
logM_TME = np.log10(M_TME)


# -------------------------------
# Kinematic statistics
# -------------------------------
v_mean = np.mean(v_los)
v_mean_mag = np.abs(v_mean)

sigma_v = np.sqrt(np.mean((v_los - v_mean)**2))


# -------------------------------
# Zero-velocity (turnaround) radius proxy
# -------------------------------
R0_pme = ((8.0 * G_cosmo * M_TME * (t_U**2)) / (np.pi**2))**(1/3)


# ============================================================
# 4. Construct input features (must match training exactly)
# ============================================================

X_M81 = np.array([
    np.log10(n_subs),
    np.log10(R0_pme),
    np.log10(M_TME),
    sigma_v,
    v_mean_mag
], dtype=np.float32).reshape(1, -1)


# ============================================================
# 5. Define the same MLP architecture used in training
# ============================================================

class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.LayerNorm(dim),
            nn.GELU(),
            nn.Dropout(0.0),
            nn.Linear(dim, dim),
            nn.LayerNorm(dim),
            nn.GELU(),
        )

    def forward(self, x):
        return x + self.block(x)


class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(5, 256),
            nn.LayerNorm(256),
            nn.GELU(),
        )
        self.layers = nn.Sequential(
            ResidualBlock(256),
            ResidualBlock(256),
            ResidualBlock(256),
        )
        self.output_proj = nn.Sequential(
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        x = self.input_proj(x)
        x = self.layers(x)
        return self.output_proj(x)


# ============================================================
# 6. Load scaler and trained model
# ============================================================

scaler = np.load("scaler_and_model_2D.31-2.npz")
X_mean = scaler["X_mean"]
X_std  = scaler["X_std"]

X_std_safe = np.where(X_std < 1e-6, 1.0, X_std)
X_M81_std = (X_M81 - X_mean) / X_std_safe

X_M81_torch = torch.tensor(X_M81_std, dtype=torch.float32)

model = MLP()
state_dict = torch.load("mlp_pme_model_2D_31-2.pt", map_location="cpu")
model.load_state_dict(state_dict)
model.eval()


# ============================================================
# 7. Predict Δlog10(M)
# ============================================================

with torch.no_grad():
    delta_pred = model(X_M81_torch).numpy().flatten()[0]

logM_corr = logM_TME + delta_pred
M_corr = 10**logM_corr


# ============================================================
# 8. Final results (clean, non-duplicated, with log-space error)
# ============================================================

LOG_ERR = 0.1075  # dex uncertainty

factor = 10**LOG_ERR
M_low  = M_corr / factor
M_high = M_corr * factor

print("\n===== Final M81 Halo Mass =====")
print(f"N_sub                 = {n_subs}")
print(f"sigma_v               = {sigma_v:.2f} km/s")

print("\nMass estimate:")
print(f"log10(M/Msun) = {logM_corr:.4f} ± {LOG_ERR:.4f} dex")
print(f"M (linear)    = {M_corr:.3e} Msun  (×/{factor:.3f}  to  ×{factor:.3f})")
print(f"Range         = [{M_low:.3e}, {M_high:.3e}] Msun")
print("================================")


===== Final M81 Halo Mass =====
N_sub                 = 31
sigma_v               = 108.57 km/s

Mass estimate:
log10(M/Msun) = 12.3855 ± 0.1075 dex
M (linear)    = 2.429e+12 Msun  (×/1.281  to  ×1.281)
Range         = [1.897e+12, 3.112e+12] Msun
